# 🎬 Film Tövsiyə Sistemi — addım-addım analiz

**Data Mining kursu üçün müstəqil layihə**

> ⚠️ **Sintetik verilənlər, real şəxslərə aid deyil.**

Bu dəftər layihənin bütün addımlarını ardıcıllıqla göstərir. Diqqət:
burada heç bir alqoritm yenidən yazılmır — dəftər `src/` qovluğundakı
**eyni modulları** çağırır. Yəni dəftərdə gördüyünüz nəticələr tətbiqdəki
nəticələrlə eynidir.

## Məzmun

1. Hazırlıq və verilənlərin oxunması
2. Verilənlərin ilkin təhlili
3. Zamana görə bölgü və sızma yoxlaması
4. Model 1 — Populyarlıq (baza)
5. Model 2 — Əməkdaşlıq süzgəci (Item-CF)
6. Model 3 — SVD (matris faktorizasiyası)
7. Üç modelin müqayisəsi
8. Gizli faktor sayının (k) təsiri
9. İzahlı tövsiyələr
10. Soyuq start analizi
11. Nəticələr

## 1. Hazırlıq və verilənlərin oxunması

Əvvəlcə layihənin kök qovluğunu Python-un axtarış yoluna əlavə edirik ki,
`src` paketini tapa bilsin.

In [1]:
import sys
from pathlib import Path

# Dəftər notebooks/ qovluğunda olduğu üçün bir səviyyə yuxarı qalxırıq
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src import evaluation as ev
from src.data_loader import load_all, film_title_map

print("Layihə qovluğu :", ROOT)
print("Təsadüfi toxum :", config.SEED)
print("numpy          :", np.__version__)
print("pandas         :", pd.__version__)

Layihə qovluğu : C:\Users\Vuzal\Desktop\SevincTeacher
Təsadüfi toxum : 42
numpy          : 2.4.2
pandas         : 3.0.1


In [2]:
customers, films, ratings = load_all()
titles = film_title_map(films)

print(f"Müştəri : {len(customers)}")
print(f"Film    : {len(films)}")
print(f"Reytinq : {len(ratings)}")

Müştəri : 150
Film    : 200
Reytinq : 5085


In [3]:
customers.head()

,customer_id,name,age_group,city,signup_date
0,1,İlkin Şirinov,55+,Bakı,2023-09-20
1,2,Kamran Nəsirov,18-24,Bakı,2022-01-17
2,3,Mələk Novruzova,45-54,Mingəçevir,2022-07-10
3,4,Kənan Zeynalov,35-44,Lənkəran,2022-08-27
4,5,Səidə Şirinova,18-24,Gəncə,2023-04-10


In [4]:
films.drop(columns=["genre_list"]).head()

,film_id,title,year,genres
0,1,Gümüşü Bulud,2004,Dram
1,2,İsti Sahil,2023,Döyüş|Dram
2,3,Şirin Xatirə,2011,Döyüş|Triller
3,4,Yüngül Körpü,2019,Animasiya
4,5,Unudulmuş Qum,1996,Sənədli|Komediya


In [5]:
ratings.head()

,customer_id,film_id,rating,timestamp
0,1,189,3,2023-12-07 08:20:46
1,1,104,3,2024-02-11 04:14:19
2,1,109,4,2024-02-14 08:56:18
3,1,186,3,2024-08-05 02:47:59
4,1,97,3,2024-09-22 19:49:48


## 2. Verilənlərin ilkin təhlili

Ən vacib göstərici **seyrəklikdir** (sparsity): matrisin neçə faizi boşdur.
Məhz bu boşluq tövsiyə sistemini zəruri edir.

In [6]:
possible = len(customers) * len(films)
sparsity = 1 - len(ratings) / possible

print(f"Mümkün xana sayı : {possible:,}".replace(",", " "))
print(f"Dolu xana sayı   : {len(ratings):,}".replace(",", " "))
print(f"Seyrəklik        : {sparsity:.2%}")
print()

per_customer = ratings.groupby("customer_id").size()
per_film = ratings.groupby("film_id").size()

print(f"Müştəri başına reytinq : min {per_customer.min()}, "
      f"orta {per_customer.mean():.1f}, maks {per_customer.max()}")
print(f"Film başına reytinq    : min {per_film.min()}, "
      f"orta {per_film.mean():.1f}, maks {per_film.max()}")

Mümkün xana sayı : 30 000
Dolu xana sayı   : 5 085
Seyrəklik        : 83.05%

Müştəri başına reytinq : min 16, orta 33.9, maks 56
Film başına reytinq    : min 1, orta 25.4, maks 97


In [7]:
# Bal paylanması
distribution = ratings["rating"].value_counts().sort_index()
summary = pd.DataFrame({
    "Reytinq sayı": distribution,
    "Pay": (distribution / len(ratings)).map("{:.2%}".format),
})
summary.index.name = "Bal"
print(f"Orta bal: {ratings['rating'].mean():.3f}")
print(f"'Bəyənilmiş' (>= {config.LIKE_THRESHOLD}) pay: "
      f"{(ratings['rating'] >= config.LIKE_THRESHOLD).mean():.2%}")
summary

Orta bal: 3.591
'Bəyənilmiş' (>= 4) pay: 54.63%


,Reytinq sayı,Pay
Bal,,
1,80,1.57%
2,575,11.31%
3,1652,32.49%
4,1818,35.75%
5,960,18.88%


In [8]:
# Uzun quyruq: ən populyar 20% film bütün reytinqlərin neçə faizini toplayır?
sorted_counts = per_film.sort_values(ascending=False)
top_20_percent = int(len(films) * 0.2)
share = sorted_counts.head(top_20_percent).sum() / sorted_counts.sum()

print(f"Ən populyar {top_20_percent} film (kataloqun 20%-i) "
      f"bütün reytinqlərin {share:.1%}-ni toplayır.")
print()
print("Ən çox reytinq alan 5 film:")
for film_id, count in sorted_counts.head(5).items():
    print(f"   {titles[film_id]:<22} {count} reytinq")

Ən populyar 40 film (kataloqun 20%-i) bütün reytinqlərin 45.4%-ni toplayır.

Ən çox reytinq alan 5 film:
   Sehrli Bağ             97 reytinq
   Qaranlıq Çay           94 reytinq
   İtirilmiş Göl          90 reytinq
   Soyuq Dağ              90 reytinq
   Sonuncu Nağıl          80 reytinq


## 3. Zamana görə bölgü və sızma yoxlaması

**Niyə zamana görə, təsadüfi deyil?** Real sistemdə biz keçmişi bilib
gələcəyi proqnozlaşdırırıq. Təsadüfi bölgüdə model müştərinin gələcək
reytinqini görüb keçmişi təxmin edə bilər — bu, süni yüksək nəticə verən
**məlumat sızmasıdır** (data leakage).

Qayda (hər müştəri üçün ayrıca):
- son **20 %** → **test** (yalnız yekun qiymətləndirmə)
- qalanın son **10 %** → **validasiya** (yalnız `k` seçimi)
- qalanı → **train**

In [9]:
from src.split import split_ratings, check_no_leakage

split = split_ratings(ratings)
split.summary()

,Dəst,Reytinq sayı,Pay
0,Train,3782,74.4%
1,Validasiya,346,6.8%
2,Test,957,18.8%


In [10]:
# Sızma yoxlaması - hamısı True olmalıdır
checks = check_no_leakage(split)
for name, passed in checks.items():
    print(f"{'✓' if passed else '✗'}  {name}: {passed}")

assert all(checks.values()), "SIZMA AŞKARLANDI!"
print()
print("Bütün yoxlamalar keçdi - test dəsti təmizdir.")

✓  train_val_kəsişmir: True


✓  train_test_kəsişmir: True
✓  val_test_kəsişmir: True
✓  train_full_düzgündür: True
✓  test_zamanca_sonuncudur: True

Bütün yoxlamalar keçdi - test dəsti təmizdir.


In [11]:
# Bir müştəri nümunəsində bölgünün necə göründüyü
example_id = int(split.test["customer_id"].iloc[0])

for label, part in [("TRAIN", split.train), ("VAL", split.val), ("TEST", split.test)]:
    subset = part[part["customer_id"] == example_id]
    print(f"{label:<6} {len(subset):>3} reytinq   "
          f"{subset['timestamp'].min():%Y-%m-%d} … {subset['timestamp'].max():%Y-%m-%d}")

TRAIN   16 reytinq   2023-12-07 … 2026-03-04
VAL      1 reytinq   2026-03-20 … 2026-03-20
TEST     4 reytinq   2026-03-25 … 2026-05-12


## 4. Model 1 — Populyarlıq (baza model)

**Gündəlik bənzətmə:** kinoteatrın girişindəki “Ən çox baxılanlar” lövhəsi.
Heç kimi tanımır, hamıya eyni siyahını verir.

İki işi var:
- **Tövsiyə:** ən çox reytinq almış filmlər
- **Bal proqnozu:** sönümlü orta (damped mean)

$$\hat{r}_i = \frac{\sum r_i + 5\mu}{n_i + 5}$$

Sönümlü orta az reytinqli filmlərin haqsız yerə yuxarı qalxmasının qarşısını
alır: 1 nəfərin 5 bal verdiyi film ümumi ortaya yaxın qalır.

In [12]:
from src.models.popularity import PopularityRecommender

# DİQQƏT: modellər YALNIZ train_full (train + validasiya) ilə öyrədilir
all_film_ids = films["film_id"].to_numpy()
popularity = PopularityRecommender().fit(split.train_full)

print(f"Ümumi orta bal: {popularity.global_mean:.4f}")
print()
print("Ən populyar 5 film:")
for film_id, count in popularity.top_films(5):
    print(f"   {titles[film_id]:<22} {count} reytinq  "
          f"(sönümlü orta {popularity.damped_means[film_id]:.2f})")

Ümumi orta bal: 3.6008

Ən populyar 5 film:
   Sehrli Bağ             78 reytinq  (sönümlü orta 3.78)
   Soyuq Dağ              76 reytinq  (sönümlü orta 3.36)
   Qaranlıq Çay           76 reytinq  (sönümlü orta 3.35)
   İtirilmiş Göl          69 reytinq  (sönümlü orta 2.99)
   Acı Pilləkən           66 reytinq  (sönümlü orta 2.59)


In [13]:
# Sönümlü ortanın təsiri: az və çox reytinqli filmlərin müqayisəsi
counts = split.train_full.groupby("film_id")["rating"].agg(["mean", "count"])
counts["sönümlü_orta"] = [popularity.damped_means[f] for f in counts.index]
counts["fərq"] = counts["sönümlü_orta"] - counts["mean"]

print("Ən az reytinqli filmlərdə düzəliş BÖYÜKDÜR:")
print(counts.nsmallest(5, "count")[["count", "mean", "sönümlü_orta", "fərq"]].round(3))
print()
print("Ən çox reytinqli filmlərdə düzəliş KİÇİKDİR:")
print(counts.nlargest(5, "count")[["count", "mean", "sönümlü_orta", "fərq"]].round(3))

Ən az reytinqli filmlərdə düzəliş BÖYÜKDÜR:


         count  mean  sönümlü_orta   fərq
film_id                                  
156          1   2.0         3.334  1.334
175          1   3.0         3.501  0.501
9            2   3.5         3.572  0.072
24           2   4.0         3.715 -0.285
27           2   3.0         3.429  0.429

Ən çox reytinqli filmlərdə düzəliş KİÇİKDİR:
         count   mean  sönümlü_orta   fərq
film_id                                   
45          78  3.795         3.783 -0.012
64          76  3.342         3.358  0.016
166         76  3.329         3.346  0.017
189         69  2.942         2.987  0.045
143         66  2.515         2.592  0.076


## 5. Model 2 — Əməkdaşlıq süzgəci (Item-based CF)

**Gündəlik bənzətmə:** “Bu filmi bəyənənlər həm də bunu bəyənib.”

Üç addım:
1. **Düzəldilmiş kosinus** — hər müştərinin balından öz ortasını çıxırıq
   (kimisi hər şeyə 5 verir, kimisi maksimum 3)
2. **Sıxılma düzəlişi** — $sim \cdot \frac{n}{n+10}$, az ortaq reytinqdə
   oxşarlığı zəiflədir
3. **Proqnoz** — ən oxşar 30 qonşunun çəkili ortası

In [14]:
from src.models.item_cf import ItemCFRecommender

item_cf = ItemCFRecommender().fit(split.train_full, all_film_ids=all_film_ids)

print(f"Oxşarlıq matrisinin ölçüsü : {item_cf.similarity.shape}")
print(f"Qonşu sayı                 : {item_cf.n_neighbors}")
print(f"Sıxılma sabiti             : {item_cf.shrinkage}")

Oxşarlıq matrisinin ölçüsü : (200, 200)
Qonşu sayı                 : 30
Sıxılma sabiti             : 10


In [15]:
# Bir filmə ən oxşar filmlər
example_film = popularity.top_films(1)[0][0]
print(f"«{titles[example_film]}» filminə ən oxşar 5 film:")
print()
for film_id, similarity in item_cf.similar_films(example_film, n=5):
    genres = films.loc[films["film_id"] == film_id, "genres"].iloc[0]
    print(f"   {titles[film_id]:<22} oxşarlıq {similarity:+.3f}   ({genres})")

«Sehrli Bağ» filminə ən oxşar 5 film:

   Kölgəli Liman          oxşarlıq +0.218   (Triller)
   Sonuncu Qala           oxşarlıq +0.168   (Komediya|Triller)
   Yüngül Dəniz           oxşarlıq +0.146   (Triller)
   Sonuncu Qatar          oxşarlıq +0.140   (Triller)
   Acı Pilləkən           oxşarlıq +0.127   (Triller)


In [16]:
# Sıxılmanın təsirini göstərək
co_counts = item_cf.mask.T @ item_cf.mask
for n in [2, 5, 10, 30, 100]:
    print(f"   {n:>3} ortaq reytinq -> oxşarlıq {n / (n + item_cf.shrinkage):.2f} "
          f"dəfəyə çevrilir")

     2 ortaq reytinq -> oxşarlıq 0.17 dəfəyə çevrilir
     5 ortaq reytinq -> oxşarlıq 0.33 dəfəyə çevrilir
    10 ortaq reytinq -> oxşarlıq 0.50 dəfəyə çevrilir
    30 ortaq reytinq -> oxşarlıq 0.75 dəfəyə çevrilir
   100 ortaq reytinq -> oxşarlıq 0.91 dəfəyə çevrilir


## 6. Model 3 — SVD (meyilli Funk SVD)

**Gündəlik bənzətmə:** musiqi pultundakı tənzimləyici düymələr. Hər filmin
bu düymələrdə müəyyən səviyyəsi, hər müştərinin isə həmin düymələrə marağı var.
Düymələrin adını **biz vermirik** — model onları verilənlərdən özü kəşf edir.

$$\hat{r}_{ui} = \mu + b_u + b_i + p_u \cdot q_i$$

SGD ilə öyrədilir: hər reytinqə baxıb xətanı hesablayır və bütün parametrləri
xətanı azaldacaq istiqamətdə kiçik addım dəyişir.

In [17]:
from src.models.svd import SVDRecommender

svd = SVDRecommender(
    n_factors=config.SVD_DEFAULT_FACTORS,
    n_epochs=config.SVD_EPOCHS,
).fit(split.train_full, all_film_ids=all_film_ids)

print(f"Gizli faktor sayı (k) : {svd.n_factors}")
print(f"Epoch sayı            : {svd.n_epochs}")
print(f"Öyrənmə addımı (lr)   : {svd.lr}")
print(f"Requlyarizasiya (reg) : {svd.reg}")
print()
print(f"P matrisi (müştəri x k) : {svd.P.shape}")
print(f"Q matrisi (film x k)    : {svd.Q.shape}")

Gizli faktor sayı (k) : 10
Epoch sayı            : 20
Öyrənmə addımı (lr)   : 0.005
Requlyarizasiya (reg) : 0.02

P matrisi (müştəri x k) : (150, 10)
Q matrisi (film x k)    : (200, 10)


In [18]:
# Təlim prosesi: hər epoch-dan sonra xəta azalır
curve = pd.DataFrame({
    "epoch": range(1, len(svd.train_rmse_history) + 1),
    "təlim_rmse": svd.train_rmse_history,
})
print(curve.head(5).to_string(index=False))
print("   ...")
print(curve.tail(3).to_string(index=False))
print()
print(f"Başlanğıc -> son: {svd.train_rmse_history[0]:.4f} -> "
      f"{svd.train_rmse_history[-1]:.4f}")

 epoch  təlim_rmse
     1    0.953756
     2    0.911325
     3    0.880439
     4    0.857533
     5    0.840163
   ...
 epoch  təlim_rmse
    18    0.770494
    19    0.768357
    20    0.766258

Başlanğıc -> son: 0.9538 -> 0.7663


In [19]:
# Öyrənilmiş meyillər nəyi göstərir?
print("Ən 'səxavətli' 3 müştəri (b_u ən yüksək):")
for pos in np.argsort(-svd.b_u)[:3]:
    cid = int(svd.imap.customer_ids[pos])
    name = customers.loc[customers['customer_id'] == cid, 'name'].iloc[0]
    print(f"   {name:<22} b_u = {svd.b_u[pos]:+.3f}")

print()
print("Ən yüksək meylli 3 film (b_i ən yüksək):")
for pos in np.argsort(-svd.b_i)[:3]:
    fid = int(svd.imap.film_ids[pos])
    print(f"   {titles[fid]:<22} b_i = {svd.b_i[pos]:+.3f}")

Ən 'səxavətli' 3 müştəri (b_u ən yüksək):
   Aysel Sadıqova         b_u = +0.834
   İlkin Vəliyev          b_u = +0.622
   Aytən Məmmədova        b_u = +0.611

Ən yüksək meylli 3 film (b_i ən yüksək):
   Unudulmuş Mahnı        b_i = +0.775
   Sehrli Qatar           b_i = +0.711
   Yalnız Küçə            b_i = +0.702


## 7. Üç modelin müqayisəsi

İndi hər üç modeli **eyni qaydalarla** test setində qiymətləndiririk:

- **RMSE / MAE** — bal proqnozunun dəqiqliyi
- **Precision@10** — təklif olunan 10 filmdən neçəsi bəyənilib
- **Recall@10** — bəyənilən filmlərin neçə faizi tutuldu
- **Kataloq əhatəsi** — kataloqun neçə faizi ümumiyyətlə təklif olunub

Namizəd filmlər: təlimdə ən azı 3 reytinqi olan və müştərinin görmədiyi filmlər.

In [20]:
models = {
    "popularity": popularity,
    "item_cf": item_cf,
    "svd": svd,
}

scores = {
    key: ev.evaluate_model(model, split.train_full, split.test,
                           catalog_size=len(all_film_ids))
    for key, model in models.items()
}

table = ev.results_table(scores).drop(columns=["model_key"])
table.round(4)

,Model,RMSE,MAE,Precision@10,Recall@10,Kataloq əhatəsi
0,Populyarlıq (baza model),0.8882,0.7181,0.0759,0.1998,0.135
1,Əməkdaşlıq süzgəci (Item-CF),0.7908,0.6222,0.0545,0.1851,0.880
2,SVD (matris faktorizasiyası),0.8456,0.6668,0.0469,0.1520,0.110


In [21]:
# Qalibləri tapaq
comparison = ev.results_table(scores)
precision_col = f"Precision@{config.TOP_N}"

best_rmse = comparison.loc[comparison["RMSE"].idxmin()]
best_precision = comparison.loc[comparison[precision_col].idxmax()]
best_coverage = comparison.loc[comparison["Kataloq əhatəsi"].idxmax()]

print(f"Bal proqnozunda ən dəqiq  : {best_rmse['Model']} "
      f"(RMSE {best_rmse['RMSE']:.4f})")
print(f"Tövsiyədə ən dəqiq        : {best_precision['Model']} "
      f"(P@10 {best_precision[precision_col]:.4f})")
print(f"Ən geniş kataloq əhatəsi  : {best_coverage['Model']} "
      f"({best_coverage['Kataloq əhatəsi']:.1%})")

Bal proqnozunda ən dəqiq  : Əməkdaşlıq süzgəci (Item-CF) (RMSE 0.7908)
Tövsiyədə ən dəqiq        : Populyarlıq (baza model) (P@10 0.0759)
Ən geniş kataloq əhatəsi  : Əməkdaşlıq süzgəci (Item-CF) (88.0%)


### 🤔 Niyə RMSE və Precision@10 fərqli modelləri seçir?

İki metrika **fərqli suallar** verir:

- **RMSE** bütün test xanalarında *bal proqnozunun* dəqiqliyini ölçür
- **Precision@10** yalnız *ilk 10 filmin sırasına* baxır

Üstəlik, offline qiymətləndirmənin **populyarlıq meyli** var: test setinə
yalnız müştərinin faktiki baxdığı filmlər düşür, insanlar isə əsasən populyar
filmlərə baxır. Aşağıdakı hesablama bunu rəqəmlə göstərir.

In [22]:
train_counts = split.train_full.groupby("film_id").size()
pool = ev.eligible_films(split.train_full)
seen = ev.seen_films_by_customer(split.train_full)

average_popularity = {}
for key, model in models.items():
    values = []
    for cid in sorted(seen)[:60]:
        candidates = np.array([f for f in pool if f not in seen[cid]])
        values += [int(train_counts.get(f, 0))
                   for f, _ in model.recommend(cid, candidates, n=10)]
    average_popularity[key] = float(np.mean(values))

print(f"Kataloqun orta reytinq sayı: {train_counts.reindex(pool).mean():.1f}")
print()
for key, value in average_popularity.items():
    print(f"   {config.MODEL_SHORT_AZ[key]:<12} tövsiyələrinin orta reytinq sayı: "
          f"{value:.1f}")

Kataloqun orta reytinq sayı: 21.4

   Populyarlıq  tövsiyələrinin orta reytinq sayı: 57.4
   Item-CF      tövsiyələrinin orta reytinq sayı: 18.0
   SVD          tövsiyələrinin orta reytinq sayı: 25.9


## 8. Gizli faktor sayının (k) təsiri

`k` **yalnız validasiya seti** ilə seçilir — test seti toxunulmaz qalır.

Burada `results/` qovluğundakı hazır nəticələri oxuyuruq (onlar
`python -m src.experiments` əmri ilə hesablanıb). Hər `k` dəyəri
**5 fərqli təsadüfi toxumla** təkrarlanıb, çünki bir tək ölçmədə SGD-nin
təsadüfiliyi `k`-lar arasındakı fərqdən böyük olur.

In [23]:
k_sweep = pd.read_csv(config.RESULTS_DIR / "k_sweep.csv", encoding=config.CSV_ENCODING)
validation = k_sweep[k_sweep["dataset"] == "validasiya"]

pivot = validation.pivot_table(index="k", columns="epochs", values="rmse")
pivot.columns = [f"{c} epoch" for c in pivot.columns]
print("Validasiya RMSE (5 toxumun ortalaması):")
pivot.round(4)

Validasiya RMSE (5 toxumun ortalaması):


,20 epoch,100 epoch
k,,
2,0.8159,0.8269
5,0.8158,0.8204
10,0.8172,0.8066
20,0.8143,0.7979
50,0.8208,0.7943


In [24]:
epoch_sweep = pd.read_csv(config.RESULTS_DIR / "epoch_sweep.csv",
                          encoding=config.CSV_ENCODING)
print("Epoch diaqnozu — 20 epoch kifayət edirmi?")
print()
print(epoch_sweep.round(4).to_string(index=False))
print()

best = epoch_sweep.loc[epoch_sweep["val_rmse"].idxmin()]
spec = epoch_sweep[epoch_sweep["epochs"] == config.SVD_EPOCHS].iloc[0]
print(f"Tapşırıq ayarı ({config.SVD_EPOCHS} epoch) : validasiya RMSE "
      f"{spec['val_rmse']:.4f}")
print(f"Ən yaxşı nöqtə ({int(best['epochs'])} epoch)  : validasiya RMSE "
      f"{best['val_rmse']:.4f}")

Epoch diaqnozu — 20 epoch kifayət edirmi?

 epochs  train_rmse  val_rmse
      5      0.8455    0.8483
     10      0.7985    0.8242
     20      0.7663    0.8178
     40      0.7128    0.8182
     60      0.6182    0.8084
    100      0.4487    0.7930
    150      0.3496    0.7863
    200      0.3012    0.7875
    300      0.2557    0.8031

Tapşırıq ayarı (20 epoch) : validasiya RMSE 0.8178
Ən yaxşı nöqtə (150 epoch)  : validasiya RMSE 0.7863


**Tapıntı:** təlim RMSE-si daim düşür (model əzbərləyir), validasiya RMSE-si
isə əvvəl düşür, sonra qalxır. Tapşırıqda tələb olunan `20 epoch, lr=0.005,
reg=0.02` dəyərləri `surprise` kitabxanasının standart ayarlarıdır və
MovieLens-100k (100 000 reytinq) kimi **20 dəfə böyük** verilənlər üçün
nəzərdə tutulub. Bizim verilənlərdə model bu nöqtədə **hələ tam öyrənməyib**.

Məhz buna görə `20 epoch` ilə `k`-nın artırılması nəticəyə təsir etmir:
model əlavə gizli faktorlardan istifadə edə biləcək qədər öyrədilməyib.

## 9. İzahlı tövsiyələr

Real sistemlərdə **izah** istifadəçi etibarını əhəmiyyətli dərəcədə artırır.
Hər model öz “düşüncə tərzini” izah edir.

In [25]:
# Test setində bəyəndiyi film olan bir müştəri seçək
liked_in_test = (
    split.test[split.test["rating"] >= config.LIKE_THRESHOLD]
    .groupby("customer_id").size().sort_values(ascending=False)
)
demo_id = int(liked_in_test.index[0])

customer_row = customers.loc[customers["customer_id"] == demo_id].iloc[0]
history = split.train_full[split.train_full["customer_id"] == demo_id]

print(f"Müştəri: {customer_row['name']} (#{demo_id}), {customer_row['city']}, "
      f"{customer_row['age_group']}")
print(f"Təlimdə {len(history)} reytinq, orta bal {history['rating'].mean():.2f}")
print()
print("Təlim dövründə ən yüksək qiymətləndirdiyi filmlər:")
top_liked = history.nlargest(5, "rating")
for row in top_liked.itertuples():
    print(f"   {titles[row.film_id]:<22} {int(row.rating)} bal")

Müştəri: Fərid Əhmədov (#7), Lənkəran, 35-44
Təlimdə 45 reytinq, orta bal 3.67

Təlim dövründə ən yüksək qiymətləndirdiyi filmlər:
   Sehrli Qatar           5 bal
   Sınıq Kənd             5 bal
   Boş Dağ                5 bal
   Qarlı Zəng             5 bal
   Ağ Külək               5 bal


In [26]:
candidates = np.array([f for f in pool if f not in seen[demo_id]])
truly_liked = set(
    split.test[(split.test["customer_id"] == demo_id)
               & (split.test["rating"] >= config.LIKE_THRESHOLD)]["film_id"]
)

print(f"Namizəd film sayı: {len(candidates)}")
print(f"Test dövründə bəyəndiyi film sayı: {len(truly_liked)}")
print("=" * 78)

for key, model in models.items():
    recommendations = model.recommend(demo_id, candidates, n=5)
    hits = sum(1 for f, _ in recommendations if f in truly_liked)
    print()
    print(f"{config.MODEL_NAMES_AZ[key]}   (tuş gələn: {hits}/5)")
    print("-" * 78)
    for rank, (film_id, score) in enumerate(recommendations, start=1):
        mark = " ✓" if film_id in truly_liked else ""
        print(f"  {rank}. {titles[film_id]}{mark}")
        print(f"     {model.explain(demo_id, film_id, titles)}")

Namizəd film sayı: 148
Test dövründə bəyəndiyi film sayı: 8

Populyarlıq (baza model)   (tuş gələn: 1/5)
------------------------------------------------------------------------------
  1. Soyuq Dağ
     Bu filmi 76 müştəri qiymətləndirib - kataloqun ən populyar filmlərindəndir.
  2. İtirilmiş Göl
     Bu filmi 69 müştəri qiymətləndirib - kataloqun ən populyar filmlərindəndir.
  3. Acı Pilləkən
     Bu filmi 66 müştəri qiymətləndirib - kataloqun ən populyar filmlərindəndir.
  4. Sonuncu Nağıl ✓
     Bu filmi 62 müştəri qiymətləndirib - kataloqun ən populyar filmlərindəndir.
  5. Yüngül Otaq
     Bu filmi 56 müştəri qiymətləndirib - kataloqun ən populyar filmlərindəndir.

Əməkdaşlıq süzgəci (Item-CF)   (tuş gələn: 1/5)
------------------------------------------------------------------------------
  1. Qaranlıq Göl
     «Boş Dağ» filminə yüksək bal verdiyiniz üçün (oxşarlıq 0.05).
  2. Dərin Bağ
     «Boş Dağ» filminə yüksək bal verdiyiniz üçün (oxşarlıq 0.11).
  3. Sonuncu Nağıl ✓
     

## 10. Soyuq start analizi

**Soyuq start problemi:** yeni qeydiyyatdan keçmiş müştəri haqqında heç nə
bilinmir. Əməkdaşlıq süzgəci və SVD isə məhz keçmiş reytinqlərdən qidalanır.

Təcrübə: 30 təsadüfi müştərinin təlim reytinqləri **3, 5 və 10** ədədə
endirilir, modellər yenidən öyrədilir və Precision@10 ölçülür.

In [27]:
cold_start = pd.read_csv(config.RESULTS_DIR / "cold_start.csv",
                         encoding=config.CSV_ENCODING)

pivot = cold_start.pivot_table(index="sira", columns="model_key",
                               values="precision_at_10")
pivot = pivot.rename(index={3: "3 reytinq", 5: "5 reytinq",
                            10: "10 reytinq", 999: "Tam təlim"})
pivot = pivot.rename(columns=config.MODEL_SHORT_AZ)
pivot.index.name = "Bilinən reytinq"
print(f"Precision@10 ({int(cold_start['musteri_sayi'].iloc[0])} müştəri üzrə):")
pivot.round(4)

Precision@10 (29 müştəri üzrə):


model_key,Item-CF,Populyarlıq,SVD
Bilinən reytinq,,,
3 reytinq,0.0034,0.0621,0.0241
5 reytinq,0.0207,0.0621,0.0172
10 reytinq,0.0276,0.0655,0.0207
Tam təlim,0.0310,0.0759,0.0276


In [28]:
# SVD-də fold-in: yeni istifadəçi üçün yalnız p_u öyrədilir, Q dondurulur
new_user_ratings = [(int(f), 5.0) for f, _ in popularity.top_films(3)]

print("Yeni istifadəçi bu 3 filmə 5 bal verdi:")
for film_id, rating in new_user_ratings:
    print(f"   {titles[film_id]}")

b_new, p_new = svd.fold_in(new_user_ratings)
print()
print(f"Öyrənilmiş meyl (b_u)     : {b_new:+.4f}")
print(f"Öyrənilmiş vektor (p_u)   : {np.round(p_new, 3)}")
print()

rated_ids = {f for f, _ in new_user_ratings}
new_candidates = np.array([f for f in pool if f not in rated_ids])
print("Fold-in ilə ilk 5 tövsiyə:")
for rank, (film_id, score) in enumerate(
        svd.recommend_foldin(new_user_ratings, new_candidates, n=5), start=1):
    print(f"   {rank}. {titles[film_id]:<22} proqnoz {score:.2f}")

Yeni istifadəçi bu 3 filmə 5 bal verdi:
   Sehrli Bağ
   Soyuq Dağ
   Qaranlıq Çay

Öyrənilmiş meyl (b_u)     : +0.8631
Öyrənilmiş vektor (p_u)   : [ 0.028 -0.061  0.07   0.086 -0.27  -0.214  0.121 -0.026 -0.084 -0.007]

Fold-in ilə ilk 5 tövsiyə:
   1. Boş Dağ                proqnoz 5.00
   2. Yalnız Küçə            proqnoz 5.00
   3. Qızılı Damla           proqnoz 5.00
   4. Sehrli Qatar           proqnoz 5.00
   5. Soyuq Bağ              proqnoz 5.00


## 11. Nəticələr

### Əsas tapıntılar

1. **Tək qalib yoxdur.** Bal proqnozunda bir model, tövsiyə siyahısında
   başqa model öndədir. “Ən yaxşı model” sualının cavabı **məqsəddən** asılıdır.

2. **Sadə baza model gözlənildiyindən güclüdür.** Populyarlıq modeli
   Precision@10-da öndədir — qismən offline qiymətləndirmənin populyarlıq
   meyli səbəbindən.

3. **Standart hiperparametrlər kiçik verilənlərə uyğun gəlmir.** 20 epoch
   bizim ~5000 reytinqlik verilənlərdə modeli tam öyrətmir.

4. **Kataloq əhatəsi dəqiqlikdən asılı deyil.** Item-CF kataloqun böyük
   hissəsini göstərir, populyarlıq modeli isə eyni bir neçə filmi təkrarlayır.
   Dəqiqlik metrikaları bu fərqi görmür.

5. **Soyuq startda sadəlik qazanır.** 3 reytinqlə populyarlıq modeli hər iki
   fərdiləşdirilmiş modeldən yaxşıdır.

### Məhdudiyyətlər

- Verilənlər **tamamilə sintetikdir** — real şəxs və davranış deyil
- Verilənlər **kiçikdir** (150 müştəri) — nəticələr dalğalıdır
- Reytinq real davranışı tam əks etdirmir
- Offline qiymətləndirmənin öz meyli var (“missing not at random”)
- Nəticələr **real bazara ümumiləşdirilə bilməz**

### Təkrarlanabilirlik

Bütün addımlarda `seed = 42`. Eyni paket versiyaları ilə eyni rəqəmlər alınır.

```
python data/generate_data.py     # verilənləri yarat
python -m src.experiments        # eksperimentləri işlət
python -m pytest tests -q        # 53 testi yoxla
streamlit run app.py             # interaktiv tətbiqi aç
```

---

> ⚠️ **Sintetik verilənlər, real şəxslərə aid deyil.**